# RAG Evaluation mit DeepEval

Dieses Notebook evaluiert die RAG-Pipeline mit **DeepEval** und **GPT-4o-mini** als Judge-Modell.

### Metriken
| Metrik | Was wird gemessen | Referenzantwort nötig? |
|---|---|---|
| **Faithfulness** | Erfindet das LLM Infos die nicht im Kontext stehen? | Nein |
| **Answer Relevancy** | Beantwortet die Antwort wirklich die Frage? | Nein |
| **Contextual Precision** | Sind die retrieved Chunks tatsächlich relevant? | Nein |
| **Contextual Recall** | Wurden alle relevanten Stellen gefunden? | Ja |

### Kategorien im Fragenkatalog
- **A** – Faktische Detail- und Datenextraktion
- **B** – Methoden- und Begründungsverständnis  
- **C** – Cross-Document-Synthese
- **D** – Struktur- und Orientierungsfragen
- **E** – Anti-Halluzination und Unsicherheitsverhalten
- **F** – Domänenwissen-Transfer

In [3]:
pip install deepeval openai python-dotenv


Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Setup: Projekt-Root setzen
import os
import sys
from pathlib import Path

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.chdir(ROOT)
print('ROOT:', ROOT)



ROOT: c:\Users\Mauricio.Guerrero\Documents\rag_project


In [2]:
# API-Key laden
from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

import openai
if not os.getenv('OPENAI_API_KEY') or os.getenv('OPENAI_API_KEY') == 'your_api_key_here':
    raise ValueError('OPENAI_API_KEY nicht gesetzt! Bitte in .env eintragen.')

print('API-Key geladen.')

API-Key geladen.


In [18]:
# Konfiguration

# RAG-Einstellungen
RAG_MODE       = 'hybrid'
RAG_K          = 20
CHUNK_SIZE     = 1200
CHUNK_OVERLAP  = 200
USE_RERANKER   = True
RERANK_TOP_N   = 5

# Judge-Modell (GPT-4o-mini ist günstiger, für Bachelorarbeit ausreichend)
JUDGE_MODEL = 'gpt-4o-mini'   # Alternativ: 'gpt-4o'

# Welche Fragen evaluieren? None = alle 37
# Beispiel für schnellen Test: QUESTION_IDS = ['q1', 'q2', 'q3']
QUESTION_IDS = None #['q1', 'q2', 'q3', 'q4', 'q5', 'q6', 'q7', 'q8'] #None

# Schwellwert ab dem eine Metrik als bestanden gilt (0.0 – 1.0)
THRESHOLD = 0.7

print(f'Judge: {JUDGE_MODEL} | RAG: {RAG_MODE}, k={RAG_K}, reranker={USE_RERANKER}, top_n={RERANK_TOP_N}')
print(f'Threshold: {THRESHOLD} | Fragen: {"alle" if QUESTION_IDS is None else QUESTION_IDS}')

Judge: gpt-4o-mini | RAG: hybrid, k=20, reranker=True, top_n=5
Threshold: 0.7 | Fragen: alle


In [19]:
# Fragen laden
from src.evaluation import load_questions

QUESTIONS_PATH = ROOT / 'data' / 'eval' / 'questions.jsonl'
all_questions = load_questions(QUESTIONS_PATH)

if QUESTION_IDS is not None:
    questions = [q for q in all_questions if q['id'] in QUESTION_IDS]
else:
    questions = all_questions

print(f'{len(questions)} Fragen geladen.')
for q in questions:
    print(f"  [{q['id']}] Kategorie {q['category']}: {q['query'][:90]}...")

print('Lese von:', QUESTIONS_PATH)
print('Datei existiert:', QUESTIONS_PATH.exists())


37 Fragen geladen.
  [q1] Kategorie A: Welche Füll- und Lösezeiten gelten für den Bremszylinder in der Bremsstellung G und P nach...
  [q2] Kategorie A: Welche Komponenten verursachten laut Peche die meisten außerplanmäßigen Instandhaltungsmaß...
  [q3] Kategorie A: Wie viele Maßnahmenvorschläge und Handlungsfelder wurden im DZSF-Bericht identifiziert?...
  [q4] Kategorie A: Welche Sensitivitätsindizes nach Sobol' werden in der Dissertation von Jobstfinke verwende...
  [q5] Kategorie A: Welche relevanten Normen werden bei der Entwicklung der automatisierten Bremsprobe nach Pe...
  [q6] Kategorie A: Welche drei Entwicklungen im Eisenbahnwesen nennt Jobstfinke als potenzielle Einflussfakto...
  [q7] Kategorie A: Welche Zuglänge in Metern und welche Art der Bremsung werden im Prüfszenario 4 zur Plausib...
  [q8] Kategorie A: Wie ist der Sensitivitätsindex erster Ordnung Si nach Sobol' mathematisch definiert und wa...
  [q9] Kategorie B: Warum reicht der Sensitivitätsindex erster Ordnung a

In [20]:
# RAG für alle Fragen ausführen und Ergebnisse sammeln
# Hinweis: Dieser Schritt läuft lokal (Ollama + Reranker) und dauert je nach
# Hardware ca. 30–120 Sekunden pro Frage.
import time
from src.evaluation import run_rag_for_eval

rag_results = []

for i, q in enumerate(questions, start=1):
    print(f'[{i}/{len(questions)}] {q["id"]} ({q["category"]}): {q["query"][:60]}...')
    t0 = time.perf_counter()

    result = run_rag_for_eval(
        query=q['query'],
        mode=RAG_MODE,
        k=RAG_K,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        use_reranker=USE_RERANKER,
        rerank_top_n=RERANK_TOP_N,
    )

    dt = time.perf_counter() - t0
    rag_results.append({
        'id': q['id'],
        'category': q['category'],
        'query': q['query'],
        'expected_output': q.get('expected_output', ''),
        'actual_output': result['answer'],
        'retrieval_context': result['retrieval_context'],
        'sources': result['sources'],
        'latency_s': round(dt, 2),
    })
    print(f'  -> {dt:.1f}s | Antwort: {result["answer"][:100]}...')

print(f'\nAlle {len(rag_results)} RAG-Läufe abgeschlossen.')

[1/37] q1 (A): Welche Füll- und Lösezeiten gelten für den Bremszylinder in ...
  -> 113.1s | Antwort: Die Füll- und Lösezeiten für den Bremszylinder in der Bremsstellung G und P sind in Tabelle 2.2 nach...
[2/37] q2 (A): Welche Komponenten verursachten laut Peche die meisten außer...
  -> 96.7s | Antwort: Die relevantesten Hinweise im Kontext sind:

* Seite 36: Die Komponenten, welche die meisten außerpl...
[3/37] q3 (A): Wie viele Maßnahmenvorschläge und Handlungsfelder wurden im ...
  -> 48.9s | Antwort: Die Information ist im bereitgestellten Dokument nicht enthalten....
[4/37] q4 (A): Welche Sensitivitätsindizes nach Sobol' werden in der Disser...
  -> 104.4s | Antwort: Die Sensitivitätsindizes nach Sobol', die in der Dissertation von Jobstfinke verwendet werden, sind ...
[5/37] q5 (A): Welche relevanten Normen werden bei der Entwicklung der auto...
  -> 136.8s | Antwort: Die relevantesten Normen bei der Entwicklung der automatisierten Bremsprobe nach [33] sind:

1. DIN ...
[6/37] 

In [21]:
# DeepEval Test Cases erstellen
from deepeval.test_case import LLMTestCase

test_cases = []
for r in rag_results:
    tc = LLMTestCase(
        input=r['query'],
        actual_output=r['actual_output'],
        expected_output=r['expected_output'],
        retrieval_context=r['retrieval_context'],
    )
    test_cases.append(tc)

print(f'{len(test_cases)} Test Cases erstellt.')

37 Test Cases erstellt.


In [22]:
# Metriken konfigurieren mit GPT-4o-mini als Judge
from deepeval.models import GPTModel
from deepeval.metrics import (
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
)

judge = GPTModel(model=JUDGE_MODEL)

metrics = [
    FaithfulnessMetric(model=judge, threshold=THRESHOLD),
    AnswerRelevancyMetric(model=judge, threshold=THRESHOLD),
    ContextualPrecisionMetric(model=judge, threshold=THRESHOLD),
    ContextualRecallMetric(model=judge, threshold=THRESHOLD),
]

print('Metriken konfiguriert:', [type(m).__name__ for m in metrics])

Metriken konfiguriert: ['FaithfulnessMetric', 'AnswerRelevancyMetric', 'ContextualPrecisionMetric', 'ContextualRecallMetric']


In [28]:
# Evaluation ausführen – in Batches um RateLimitError zu vermeiden
import time
from deepeval import evaluate
from deepeval.evaluate.configs import AsyncConfig

BATCH_SIZE = 7        # Fragen pro Batch
SLEEP_BETWEEN = 10    # Sekunden Pause zwischen Batches

# run_async=False: sequenziell statt parallel -> kein Rate Limit Druck
# throttle_value=5: zusätzlich 5s Pause zwischen Anfragen
async_cfg = AsyncConfig(run_async=False, throttle_value=5)

all_test_results = []

for i in range(0, len(test_cases), BATCH_SIZE):
    batch = test_cases[i:i + BATCH_SIZE]
    batch_num = i // BATCH_SIZE + 1
    total_batches = (len(test_cases) + BATCH_SIZE - 1) // BATCH_SIZE
    print(f'Batch {batch_num}/{total_batches} ({len(batch)} Fragen)...')

    result = evaluate(batch, metrics=metrics, async_config=async_cfg)
    all_test_results.extend(result.test_results)

    if i + BATCH_SIZE < len(test_cases):
        print(f'  Pause {SLEEP_BETWEEN}s...')
        time.sleep(SLEEP_BETWEEN)

class _FakeEvalResult:
    def __init__(self, test_results):
        self.test_results = test_results

eval_results = _FakeEvalResult(all_test_results)
print(f'\nEvaluation abgeschlossen. {len(all_test_results)} Ergebnisse.')


Batch 1/6 (7 Fragen)...


✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4o-mini, strict=False, async_mode=False)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4o-mini, strict=False, async_mode=False)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using gpt-4o-mini, strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using gpt-4o-mini, strict=False, 
async_mode=False)...

...

✨ You're running DeepEval's latest Contextual Recall Metric! (using gpt-4o-mini, strict=False, 
async_mode=False)...



Metrics Summary

  - ❌ Faithfulness (score: 0.4, threshold: 0.7, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.40 because the actual output includes information that is not present in the provided document, as indicated by the contradictions stating that the information is not included and that partial verification is generally not recommended., error: None)
  - ❌ Answer Relevancy (score: 0.6, threshold: 0.7, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.60 because there are irrelevant statements indicating that the information needed to answer the question is not present, which detracts from the overall relevance of the response., error: None)
  - ✅ Contextual Precision (score: 0.8333333333333333, threshold: 0.7, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.83 because the relevant nodes are ranked higher than the irrelevant nodes, with the first and third nodes providing direct insights into how update systems mai

⚠ WARNING: No hyperparameters logged.
» ]8;id=188923;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 83.12s | token cost: 0.004763099999999999 USD)
» Test Results (2 total tests):
   » Pass Rate: 50.0% | Passed: 1 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.


Evaluation abgeschlossen. 37 Ergebnisse.


In [29]:
# Ergebnisse pro Frage anzeigen
import pandas as pd

rows = []
for r, tc in zip(rag_results, eval_results.test_results):
    row = {
        'ID': r['id'],
        'Kategorie': r['category'],
        'Frage (kurz)': r['query'][:55] + '...',
        'Latenz (s)': r['latency_s'],
    }
    for metric_data in tc.metrics_data:
        name = metric_data.name  # z.B. "Faithfulness", "Answer Relevancy" etc.
        row[name] = round(metric_data.score, 2) if metric_data.score is not None else None
        row[name + ' ✓'] = '✓' if metric_data.success else '✗'
    rows.append(row)

df = pd.DataFrame(rows)
print(df.to_string(index=False))

 ID Kategorie                                               Frage (kurz)  Latenz (s)  Faithfulness Faithfulness ✓  Answer Relevancy Answer Relevancy ✓  Contextual Precision Contextual Precision ✓  Contextual Recall Contextual Recall ✓
 q1         A Welche Füll- und Lösezeiten gelten für den Bremszylinde...      113.15          0.80              ✓              1.00                  ✓                  0.87                      ✓                1.0                   ✓
 q2         A Welche Komponenten verursachten laut Peche die meisten ...       96.69          1.00              ✓              0.50                  ✗                  1.00                      ✓                1.0                   ✓
 q3         A Wie viele Maßnahmenvorschläge und Handlungsfelder wurde...       48.86          0.00              ✗              0.00                  ✗                  1.00                      ✓                1.0                   ✓
 q4         A Welche Sensitivitätsindizes nach Sobol' werden

In [30]:
# Begründungen anzeigen — warum hat das Modell diese Note gegeben?
# Optional: FILTER_ID = 'q1'  um nur eine bestimmte Frage zu sehen
FILTER_ID = None   # None = alle Fragen anzeigen

for r, tc in zip(rag_results, eval_results.test_results):
    if FILTER_ID and r['id'] != FILTER_ID:
        continue

    print(f"\n{'='*70}")
    print(f"[{r['id']}] Kategorie {r['category']}")
    print(f"Frage:    {r['query']}")
    print(f"Antwort:  {r['actual_output'][:200]}...")
    print()

    for metric_data in tc.metrics_data:
        status = '✓' if metric_data.success else '✗'
        score  = round(metric_data.score, 2) if metric_data.score is not None else '–'
        print(f"  {status} {metric_data.name}: {score}")
        if metric_data.reason:
            print(f"     Grund: {metric_data.reason}")
        print()


[q1] Kategorie A
Frage:    Welche Füll- und Lösezeiten gelten für den Bremszylinder in der Bremsstellung G und P nach UIC 540?
Antwort:  Die Füll- und Lösezeiten für den Bremszylinder in der Bremsstellung G und P sind in Tabelle 2.2 nach UIC 540 aufgeführt.

Für die Bremsstellung G gelten folgende Werte:
- Füllzeit: 18 s - 30 s
- Lösez...

  ✓ Faithfulness: 0.8
     Grund: The score is 0.80 because the actual output incorrectly states the fill time for the brake position P as 3 s - 6 s, while the retrieval context specifies it should be 3 s - 5 s.

  ✓ Answer Relevancy: 1.0
     Grund: The score is 1.00 because the response directly addresses the question about the filling and release times for the brake cylinder in the braking positions G and P according to UIC 540, with no irrelevant statements present.

  ✓ Contextual Precision: 0.87
     Grund: The score is 0.87 because the relevant nodes are well-ranked, with the first two nodes providing direct answers to the question about Füll

In [31]:
# Zusammenfassung nach Kategorie
metric_cols = [c for c in df.columns if not c.endswith('✓') and c not in ('ID', 'Kategorie', 'Frage (kurz)', 'Latenz (s)')]
pass_cols   = [c for c in df.columns if c.endswith('✓')]

summary_by_category = df.groupby('Kategorie')[metric_cols].mean().round(2)
summary_total = df[metric_cols].mean().round(2).to_frame(name='Gesamt').T

pass_rates = {}
for col in pass_cols:
    passed = (df[col] == '✓').sum()
    pass_rates[col.replace(' ✓', '')] = f"{passed}/{len(df)} ({100*passed//len(df)}%)"
summary_pass = pd.DataFrame([pass_rates], index=['Bestandsrate'])

print(f'=== Durchschnitt nach Kategorie (Threshold={THRESHOLD}) ===')
print(summary_by_category.to_string())
print(f'\n=== Gesamt-Durchschnitt ===')
print(summary_total.to_string())
print(f'\n=== Bestandsrate ===')
print(summary_pass.to_string())

# Speichern
eval_dir = ROOT / 'data' / 'eval'
summary_by_category.to_csv(eval_dir / 'summary_by_category.csv', encoding='utf-8-sig')
summary_total.to_csv(eval_dir / 'summary_total.csv', encoding='utf-8-sig')
summary_pass.to_csv(eval_dir / 'summary_passrate.csv', encoding='utf-8-sig')
print('\nGespeichert: summary_by_category.csv, summary_total.csv, summary_passrate.csv')

=== Durchschnitt nach Kategorie (Threshold=0.7) ===
           Faithfulness  Answer Relevancy  Contextual Precision  Contextual Recall
Kategorie                                                                         
A                  0.75              0.77                  0.95               0.91
B                  0.87              0.84                  0.89               0.98
C                  0.90              0.84                  0.42               0.86
D                  0.77              0.60                  0.38               0.75
E                  0.65              0.49                  0.00               0.80
F                  0.80              0.70                  0.87               1.00

=== Gesamt-Durchschnitt ===
        Faithfulness  Answer Relevancy  Contextual Precision  Contextual Recall
Gesamt           0.8              0.73                  0.64               0.89

=== Bestandsrate ===
             Faithfulness Answer Relevancy Contextual Precision Contextua

In [36]:
# Ergebnisse speichern
import json

eval_dir = ROOT / 'data' / 'eval'
eval_dir.mkdir(parents=True, exist_ok=True)

# Tabelle als CSV (zum Öffnen in Excel)
csv_path = eval_dir / 'results.csv'
df.to_csv(csv_path, index=False, encoding='utf-8-sig')
print('CSV gespeichert:', csv_path)

# Vollständige Ergebnisse inkl. Begründungen + Top-Chunks als JSON
full_results = []
for r, tc in zip(rag_results, eval_results.test_results):
    # sources und retrieval_context haben denselben Index (beide kommen aus docs)
    chunks_with_source = []
    for i, chunk_text in enumerate(r['retrieval_context']):
        src = r['sources'][i] if i < len(r['sources']) else {}
        source_file = src.get('source') or '–'
        page = src.get('page')
        chunks_with_source.append({
            'rank': i + 1,
            'text': chunk_text,
            'source': source_file,
            'page': page,
        })

    entry = {
        'id': r['id'],
        'category': r['category'],
        'query': r['query'],
        'expected_output': r['expected_output'],
        'actual_output': r['actual_output'],
        'latency_s': r['latency_s'],
        'top_chunks': chunks_with_source,
        'metrics': [
            {
                'name': m.name,
                'score': round(m.score, 4) if m.score is not None else None,
                'success': m.success,
                'reason': m.reason,
            }
            for m in tc.metrics_data
        ],
    }
    full_results.append(entry)

json_path = eval_dir / 'results_full.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(full_results, f, ensure_ascii=False, indent=2)
print('JSON gespeichert:', json_path)


CSV gespeichert: c:\Users\Mauricio.Guerrero\Documents\rag_project\data\eval\results.csv
JSON gespeichert: c:\Users\Mauricio.Guerrero\Documents\rag_project\data\eval\results_full.json


In [37]:
# Markdown-Report aus results_full.json erzeugen
import json
from pathlib import Path

eval_dir = ROOT / 'data' / 'eval'
json_path = eval_dir / 'results_full.json'

with open(json_path, 'r', encoding='utf-8') as f:
    full_results = json.load(f)

lines = ['# RAG Evaluation Report\n']
lines.append(f'**Anzahl Fragen:** {len(full_results)}  \n')
lines.append(f'**Judge-Modell:** {JUDGE_MODEL}  \n')
lines.append(f'**RAG-Modus:** {RAG_MODE} | k={RAG_K} | Reranker={USE_RERANKER} | top_n={RERANK_TOP_N}  \n')
lines.append(f'**Threshold:** {THRESHOLD}\n\n')
lines.append('---\n')

for entry in full_results:
    lines.append(f"## [{entry['id']}] Kategorie {entry['category']}\n")
    lines.append(f"**Frage:** {entry['query']}\n\n")

    lines.append(f"**Generierte Antwort:**\n> {entry['actual_output'].strip()}\n\n")

    expected = entry.get('expected_output', '').strip()
    if expected:
        lines.append(f"**Erwartete Antwort:**\n> {expected}\n\n")

   # Metriken
   #lines.append('**Metriken:**\n\n')
   # lines.append('| Metrik | Score | Bestanden | Begründung |\n')
   # lines.append('|--------|-------|-----------|------------|\n')
   # for m in entry.get('metrics', []):
   #     score = f"{m['score']:.2f}" if m['score'] is not None else '–'
   #     status = '✓' if m['success'] else '✗'
   #     reason = (m.get('reason') or '').replace('\n', ' ').replace('|', '\\|')
   #     lines.append(f"| {m['name']} | {score} | {status} | {reason} |\n")
   # lines.append('\n')

    # Top Chunks mit Quelldokument
    top_chunks = entry.get('top_chunks', [])[:5]
    if top_chunks:
        lines.append('**Top Chunks (Retrieval Context):**\n\n')
        for chunk in top_chunks:
            src = chunk.get('source') or '–'
            page = chunk.get('page')
            src_label = Path(src).name if src != '–' else '–'
            page_label = f', Seite {page}' if page is not None else ''
            summary = f"Chunk {chunk['rank']} — {src_label}{page_label}"
            lines.append(f"<details><summary>{summary}</summary>\n\n")
            lines.append(f"```\n{chunk['text'].strip()}\n```\n\n")
            lines.append('</details>\n\n')

    lines.append('---\n\n')

md_path = eval_dir / 'results_report.md'
with open(md_path, 'w', encoding='utf-8') as f:
    f.writelines(lines)

print(f'Markdown-Report gespeichert: {md_path}')


Markdown-Report gespeichert: c:\Users\Mauricio.Guerrero\Documents\rag_project\data\eval\results_report.md
